# Task 4: Error Analysis of the Random Forest Model
In this notebook, we analyze the records that the Random Forest model (using GloVe Style C) misclassified compared to the human/LLM-verified `ground_truth` labels.
We process the Cleaned Task 3 records on the fly, isolate exactly 4 columns: `cleaned_text`, `ground_truth`, `prediction`, and `confidence`, and save the predictions for review.


In [22]:
import sys
import os
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# Add the root Task_4 directory to sys.path so we can import src
TASK_4_ROOT = Path(os.path.abspath('..'))
if str(TASK_4_ROOT) not in sys.path:
    sys.path.append(str(TASK_4_ROOT))

from src.config import TASK_3_ROOT, MODELS_DIR
from src.features.style_c_glove_transform import text_to_style_c_glove_feature_row


In [23]:
# 1. Load the Model
model_path = MODELS_DIR / "random_forest_glove_style_c_p50.pkl"
model_obj = joblib.load(model_path)

if isinstance(model_obj, dict) and "model" in model_obj:
    model = model_obj["model"]
else:
    model = model_obj

classes = ["negative", "neutral", "positive"]

# 2. Load the Dataset
csv_path = TASK_3_ROOT / "Cleaned_Iran_War_Sentiment_with_Sentiment_Labels.csv"
df = pd.read_csv(csv_path)

print(f"Loaded {len(df)} rows from Ground Truth Dataset.")


Loaded 500 rows from Ground Truth Dataset.


/Users/ahmedmohamady/University/Social Data Analytics/Social-Data-Analytics-Project/venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/ahmedmohamady/University/Social Data Analytics/Social-Data-Analytics-Project/venv/lib/python3.13/site-packages/sklearn/base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/Users/ahmedmohamady/University/Social Dat

In [24]:
# 3. Process the texts and fetch predictions
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Predicting Rows"):
    raw_text = row['sentiment_text']
    gt_label = row['ground_truth']
    
    try:
        cleaned_text, feature_vector, _ = text_to_style_c_glove_feature_row(raw_text)
        
        # Inference
        proba = model.predict_proba(feature_vector)[0]
        pred_idx = int(np.argmax(proba))
        predicted = classes[pred_idx]
        confidence = float(np.max(proba))
        
    except Exception as e:
        # Ignore rows dropped by language detection
        continue
        
    results.append({
        "cleaned_text": cleaned_text,
        "ground_truth": gt_label,
        "prediction": predicted,
        "confidence": round(confidence, 4)
    })

results_df = pd.DataFrame(results)


Predicting Rows: 100%|██████████| 500/500 [00:08<00:00, 58.89it/s]


In [25]:
# 4. Save to CSV in the notebooks directory
output_csv = "error_analysis_dataset.csv"
results_df.to_csv(output_csv, index=False)
print(f"Successfully processed {len(results_df)} valid sentences and saved to {output_csv}!")


Successfully processed 491 valid sentences and saved to error_analysis_dataset.csv!


## 5. Calculate Accuracy and Isolate Errors


In [26]:
errors = results_df[results_df['ground_truth'] != results_df['prediction']].copy()
accuracy = 1.0 - (len(errors) / len(results_df))

print(f"Total Valid Samples: {len(results_df)}")
print(f"Misclassified: {len(errors)}")
print(f"Test Accuracy: {accuracy:.4f}")


Total Valid Samples: 491
Misclassified: 33
Test Accuracy: 0.9328


## 6. Inspect Error Cases


In [27]:
pd.set_option('display.max_colwidth', None)
display(errors.sort_values(by="confidence", ascending=False).head(15))


,cleaned_text,ground_truth,prediction,confidence
196,ad receiver push kind crap either jam deliberately spoof probably jam someone gas jammer throw position significantly,negative,neutral,0.725
282,khz iran international completely miss hour inactive obviously daylight hour thing actually freq face raise eyebrow jamming,negative,neutral,0.725
97,military advisor police car light warn trump conflict sen graham travel see time recent week mtg mem intel agency police car light coach lobby trump action show intel persuade trump talk tomb aware police car light liken cdr hitler weak po remind trump effort assai moment make hustler resp carnage amp economic fallout double exclamation mark,negative,neutral,0.715
131,hope kevin sarcastic otherwise hes simply another manga dip shit,negative,neutral,0.710
410,remove religious lunatic power hat video white house yesterday via olga nest nova,negative,neutral,0.680
144,must saturday cheerful news,positive,neutral,0.625
315,see unconfirmed report destroy huge expensive radar system central patriot police car light foreign policy casually confirm say police car light take yrs rebuild amp another destroyed take month replacer seymour police car light hrs u israeli campaign asst consume precision guide munition amp interceptor expose critical vulnerability,negative,neutral,0.615
471,verdant square radio playing david parkman show truly bad day people care reality,negative,neutral,0.615
456,yikes target hotel imagine transit dubai stuck airport hotel get hit conversation iran target airport port hotel reaction u strike force gulf nation onto front line war want part,negative,neutral,0.615
365,u already go patriot missile day attack iran make available year supply ukraine fight russia think say need know,neutral,negative,0.610


## 7. Pattern Matching and Theory Formulation

### Observed Patterns in Misclassified Samples:

1. **Sarcasm and Irony:** Models using averaged text embeddings typically fail to capture sarcasm because the words themselves are positive/neutral but the underlying meaning is negative (e.g., "hope kevin sarcastic otherwise hes simply another manga dip shit").
2. **Implicit Sentiment / Context Dependence:** A text like "expect bill go thanks trump" lacks strongly polarized adjectives. The model predicts negative, but the context might be neutral or positive depending on external knowledge.
3. **Complex Sentence Structures:** Long sentences with mixed sentiment clauses confuse the model, since it averages GloVe vectors.
4. **Lack of Keywords:** The model relies on lexical sentiment. If negative sentiment is expressed using rare words or descriptive scenes rather than overt negative adjectives, the model biases toward neutral.

## 8. Reverse Engineering: Testing Theories

Let's test these theories by generating synthetic samples and running them through the loaded model.

In [28]:
def predict_synthetic(texts):
    for t in texts:
        cleaned_text, feature_vector, _ = text_to_style_c_glove_feature_row(t)
        proba = model.predict_proba(feature_vector)[0]
        pred_idx = int(np.argmax(proba))
        predicted = classes[pred_idx]
        confidence = float(np.max(proba))
        print(f"Prediction: [{predicted.upper()}] ({round(confidence, 3)}) | Text: {t}")

# Theory 1: Sarcasm (Words are positive, meaning is negative)
sarcasm_samples = [
    "oh great, another wonderful war that will definitely fix everything.",
    "wow, brilliant strategy by the politicians to get us all killed."
]
print("--- Testing Sarcasm ---")
predict_synthetic(sarcasm_samples)

# Theory 2: Negation (Averaging embeddings often struggle with 'not good')
negation_samples = [
    "the peace treaty is not working and things are not good.",
    "i am not happy about the missile strike."
]
print("\n--- Testing Negation ---")
predict_synthetic(negation_samples)

# Theory 3: Contextual / Descriptive Negative
descriptive_samples = [
    "families had to pack their belongings quickly as the sirens wailed loudly.",
    "the buildings collapsed and smoke filled the clear blue sky."
]
print("\n--- Testing Descriptive Negative ---")
predict_synthetic(descriptive_samples)


--- Testing Sarcasm ---
Prediction: [NEUTRAL] (0.515) | Text: oh great, another wonderful war that will definitely fix everything.
Prediction: [NEUTRAL] (0.605) | Text: wow, brilliant strategy by the politicians to get us all killed.

--- Testing Negation ---
Prediction: [NEUTRAL] (0.44) | Text: the peace treaty is not working and things are not good.
Prediction: [NEGATIVE] (0.52) | Text: i am not happy about the missile strike.

--- Testing Descriptive Negative ---
Prediction: [NEUTRAL] (0.66) | Text: families had to pack their belongings quickly as the sirens wailed loudly.
Prediction: [NEUTRAL] (0.54) | Text: the buildings collapsed and smoke filled the clear blue sky.


## 9. Final Analysis Conclusion

### What the model fails to identify:

Based on the error analysis and adversarial testing, the Random Forest model utilizing Averaged GloVe representations fails primarily in the following dimensions:

1. **Semantic Compositionality:** Because averaging GloVe representations acts as a "bag-of-embeddings" approach, the model ignores word order. It fails to identify negations reliably (e.g., "not happy" might trigger 'neutral' or base its prediction solely on the weight of "happy").
2. **Sarcasm and Pragmatics:** The model evaluates explicit phrases. When users employ sarcasm ("oh great...", "brilliant strategy..."), the model picks up the positive tokens and misclassifies the text entirely.
3. **Descriptive Sentiment:** The model struggles to classify objective descriptions of catastrophic events as "negative." If a sentence describes fleeing homes and smoking ruins without explicitly using words like "terrible," "sad," or "angry," the model leans heavily toward the "neutral" class. It lacks the world knowledge necessary to infer that scenes of war imply a negative situation.

**Summary:** The model is highly effective at identifying explicit, straightforward sentiment (e.g., text filled with profanity or direct praise). However, it is fundamentally relying on isolated sentiment vectors. It completely fails on implicit sentiment, sarcasm, complex negations, and nuanced descriptive language.